# Momentum 12-1 + GARCH — notebook completo

Reúne em um só arquivo os três códigos do projeto, **na ordem em que dependem uns dos outros**:

| Parte | Origem | O que faz |
|---|---|---|
| **1. Funções do momentum** | `funcoes_momentum.py` | Seleção Top N / deadband, backtest com execução defasada, custos e métricas (as funções do `momentums.ipynb`) |
| **2. Dados, volatilidade e pesos** | `initial_test.py` | Leitura da planilha, CDI, previsão de volatilidade (GARCH / GJR / EWMA) e pesos iguais ou por 1/σ̂ |
| **3. Otimização** | `momentum-garch-otimizacao.py` | Grade de 150 configurações, escolha no treino e relatório na validação |
| **4–5. Execução** | (células novas) | Rodam uma configuração e a grade completa |

## Como usar

1. Instale as dependências (célula abaixo) e deixe a planilha `economatica-3.xlsx` em uma pasta `baseDeDados/` (ou `bases de dados/`) na pasta do notebook **ou em qualquer pasta acima**. Se estiver em outro lugar, informe o caminho na célula **Parâmetros de execução**.
2. Execute as células **de cima para baixo** (*Run All* funciona). As partes 1 a 3 só definem funções e rodam em segundos.
3. Tempos da **primeira** execução: parte 4 (uma configuração) cerca de 1 a 2 minutos; parte 5 (grade completa) cerca de 5 a 10 minutos. A previsão de volatilidade fica em cache (`cache_vol_mensal.pkl`), então as próximas execuções levam segundos.
4. Arquivos criados na pasta do notebook: `cache_vol_mensal.pkl`, `cdi_sgs12.csv`, `momentum_garch_resultado.png`, `otimizacao_melhores.png`, `otimizacao_grade.csv`, `otimizacao_familias.csv`.

## Protocolo

Dados até **2021-12-31**: treino 2017–2020 e validação 2021. Os parâmetros são escolhidos só no treino. Estender `FIM_DADOS` passa a usar o período 2022+ que o protocolo reserva como teste.

## Observações

Algumas alterações em relação aos arquivos originais foram feitas para adaptá-los ao notebook.

In [ ]:
# Dependências (descomente na primeira vez):
# %pip install arch pandas numpy openpyxl matplotlib requests

%matplotlib inline
from IPython.display import display, Image

---
## Parte 1 — Funções do momentum (`funcoes_momentum.py`)

Metodologia do momentum, idêntica à do `momentums.ipynb`. Nada aqui usa volatilidade.

| Função | O que faz |
|---|---|
| `preparar_precos` | Normaliza o cabeçalho, corta o período, exige preços positivos e preenche **no máximo 1** dia ausente |
| `criar_top_n` | Carteira mensal com os *N* primeiros do ranking de momentum |
| `criar_deadband` | Mantém 5 ativos; só vende quem cai para fora do top *L* (7 ou 10); as vagas vão para os melhores do ranking |
| `criar_pesos` | Pesos iguais (1/N) |
| `backtest` | Ordem no fechamento do pregão seguinte ao sinal; pesos que derivam com os retornos; turnover registrado no dia da ordem |
| `aplicar_custos` | Retorno líquido: `(1 + r) · (1 − custo · turnover) − 1` |
| `calcular_sharpe` | Sharpe anualizado (sobre o CDI, quando fornecido) |
| `calcular_metricas` | Retorno anual, volatilidade, Sharpe, drawdown máximo e turnover anual |

In [ ]:
import numpy as np
import pandas as pd


def preparar_precos(base, inicio, fim):
    precos = base.copy()
    precos.columns = ['Data'] + [str(c).strip().split('\n')[-1].strip()
                                for c in precos.columns[1:]]
    if precos.columns.duplicated().any():
        raise ValueError('Tickers duplicados após normalização do cabeçalho.')
    precos['Data'] = pd.to_datetime(precos['Data'], errors='coerce')
    precos = precos.dropna(subset=['Data']).set_index('Data').sort_index()
    precos = precos.loc[inicio:fim]
    if precos.index.has_duplicates:
        raise ValueError('Datas duplicadas na planilha.')
    precos = precos.replace('-', np.nan).apply(pd.to_numeric, errors='coerce')
    precos = precos.dropna(how='all').ffill(limit=1)
    if precos.empty or (precos <= 0).any().any() or np.isinf(precos.to_numpy()).any():
        raise ValueError('Base vazia ou com preços não positivos/infinitos.')
    return precos


def criar_top_n(momentum_mensal, n):
    if not isinstance(n, int) or n < 1:
        raise ValueError('n deve ser inteiro positivo.')
    historico = {}
    for data, linha in momentum_mensal.iterrows():
        ranking = linha.dropna().sort_values(ascending=False, kind='stable')
        if len(ranking) >= n:
            historico[data] = ranking.head(n).index.tolist()
    return pd.DataFrame.from_dict(historico, orient='index',
                                  columns=[f'Ativo {i}' for i in range(1, n + 1)])


def criar_deadband(momentum_mensal, limite, n=5):
    if not isinstance(n, int) or not isinstance(limite, int) or n < 1 or limite < n:
        raise ValueError('Use limite inteiro >= n >= 1.')
    carteira, historico = [], {}
    for data, linha in momentum_mensal.iterrows():
        ranking = linha.dropna().sort_values(ascending=False, kind='stable')
        if len(ranking) < n:
            continue
        permitidos = ranking.head(limite).index
        carteira = [ativo for ativo in carteira if ativo in permitidos]
        for ativo in ranking.index:
            if len(carteira) == n:
                break
            if ativo not in carteira:
                carteira.append(ativo)
        historico[data] = carteira.copy()
    return pd.DataFrame.from_dict(historico, orient='index',
                                  columns=[f'Ativo {i}' for i in range(1, n + 1)])


def criar_pesos(carteira, universo):
    pesos = pd.DataFrame(0.0, index=carteira.index, columns=universo)
    for data, linha in carteira.iterrows():
        ativos = linha.dropna().tolist()
        if not ativos or len(set(ativos)) != len(ativos):
            raise ValueError(f'Carteira vazia ou com duplicatas em {data}.')
        if not set(ativos).issubset(pesos.columns):
            raise ValueError(f'Ativo fora do universo em {data}.')
        pesos.loc[data, ativos] = 1.0 / len(ativos)
    return pesos


def backtest(pesos_alvo, retornos, inicio, fim):
    r_periodo = retornos.loc[inicio:fim]
    if r_periodo.empty or not retornos.index.is_monotonic_increasing or retornos.index.has_duplicates:
        raise ValueError('Retornos vazios ou datas fora de ordem/duplicadas.')
    if not pesos_alvo.columns.equals(retornos.columns):
        raise ValueError('Colunas dos pesos e retornos devem coincidir.')
    if pesos_alvo.index.has_duplicates or not pesos_alvo.index.isin(r_periodo.index).all():
        raise ValueError('Sinais duplicados ou fora dos pregões do backtest.')
    if (not np.isfinite(pesos_alvo.to_numpy()).all() or (pesos_alvo < 0).any().any()
            or not np.allclose(pesos_alvo.sum(axis=1), 1)):
        raise ValueError('Pesos devem ser finitos, não negativos e somar um.')
    ordens = pesos_alvo.reindex(r_periodo.index).shift(1)
    pesos_atuais = pd.Series(0.0, index=retornos.columns)
    historico = []
    for data, r in r_periodo.iterrows():
        faltantes = r.index[(pesos_atuais > 0) & r.isna()].tolist()
        if faltantes:
            raise ValueError(f'Retorno ausente em {data.date()}: {faltantes}')
        r = r.fillna(0.0)
        if not np.isfinite(r.to_numpy()).all() or (r < -1).any():
            raise ValueError(f'Retorno inválido em {data}.')
        retorno_dia = float((pesos_atuais * r).sum())
        if retorno_dia <= -1:
            raise ValueError(f'Patrimônio esgotado em {data}.')
        pesos_atuais = pesos_atuais * (1 + r) / (1 + retorno_dia)
        turnover = 0.0
        if ordens.loc[data].notna().all():
            novos_pesos = ordens.loc[data]
            turnover = float((novos_pesos - pesos_atuais).abs().sum())
            pesos_atuais = novos_pesos.copy()
        historico.append({'Data': data, 'Retorno': retorno_dia, 'Turnover': turnover})
    return pd.DataFrame(historico).set_index('Data')


def aplicar_custos(resultado, custo):
    if not np.isfinite(custo) or custo < 0 or (custo * resultado['Turnover'] >= 1).any():
        raise ValueError('Custo inválido ou superior ao patrimônio.')
    return (1 + resultado['Retorno']) * (1 - custo * resultado['Turnover']) - 1


def calcular_sharpe(retornos, pregoes_ano=252, cdi=None):
    if cdi is not None:
        rf = cdi.reindex(retornos.index, method="ffill")
        if rf.isna().any():
            raise ValueError("CDI ausente em algumas datas")
        excesso = retornos - rf
    else:
        excesso = retornos
    vol = excesso.std(ddof=1)
    if not np.isfinite(vol) or vol<= 1e-15:
        return np.nan
    return excesso.mean() / vol * np.sqrt(pregoes_ano)


def calcular_metricas(resultado, custo=0.001, pregoes_ano=252, cdi=None):
    if resultado.empty or resultado[['Retorno', 'Turnover']].isna().any().any():
        raise ValueError('Resultado vazio ou com valores ausentes.')
    r = aplicar_custos(resultado, custo)
    patrimonio = (1 + r).cumprod()
    retorno_anual = patrimonio.iloc[-1] ** (pregoes_ano / len(r)) - 1
    pico = patrimonio.cummax().clip(lower=1.0)
    return {'Retorno anual (%)': retorno_anual * 100,
            'Volatilidade (%)': r.std(ddof=1) * np.sqrt(pregoes_ano) * 100,
            'Sharpe': calcular_sharpe(r, pregoes_ano, cdi=cdi),
            'Max Drawdown (%)': (patrimonio / pico - 1).min() * 100,
            'Turnover anual': resultado['Turnover'].sum() * pregoes_ano / len(r)}

---
## Parte 2 — Dados, volatilidade e pesos (`initial_test.py`)

Descrição original do arquivo:

```text
Momentum 12-1 (Top N / Deadband) com pesos por volatilidade prevista (GARCH).

METODOLOGIA DO MOMENTUM = a do notebook momentums.ipynb, importada de funcoes_momentum.py
(funções extraídas literalmente: preparar_precos, criar_top_n, criar_deadband, criar_pesos,
backtest, aplicar_custos, calcular_sharpe, calcular_metricas). Isso evita divergência:
  - sinal: P[t-21] / P[t-L] - 1, no último pregão do mês (L = 126, 189 ou 252: "6-1","9-1","12-1")
  - seleção: ranking cross-sectional (Top N ou Deadband), carteira só comprada
  - execução: ordem no fechamento do pregão seguinte; pesos "derivam" com os retornos
  - custos: taxa sobre o turnover (compras + vendas); Sharpe com CDI quando fornecido

O QUE ESTE ARQUIVO ACRESCENTA (o GARCH):
  - previsão de volatilidade de cada ativo na data do sinal (GARCH, GJR ou EWMA), usando
    apenas retornos até o fechamento do dia do sinal;
  - pesos da carteira selecionada proporcionais a 1/σ̂ (soma 1), em vez de pesos iguais.
    A seleção dos ativos é IDÊNTICA nos dois esquemas, então a diferença de desempenho
    isola o efeito de dimensionar por volatilidade.
  - Como os pesos somam 1 (sem caixa nem alavancagem), a função backtest do notebook é
    usada sem nenhuma modificação.

PROTOCOLO: por padrão, só dados até 2021-12-31 (treino 2017–2020, validação 2021), como no
notebook. Estender FIM_DADOS usa dados que o notebook reserva como fora da amostra.

Dependências: pip install arch pandas numpy openpyxl matplotlib requests
```

### Imports
Bibliotecas usadas nesta parte (as funções de `funcoes_momentum` já foram definidas na Parte 1).

In [ ]:
from __future__ import annotations

import os
import pickle
import re
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from arch import arch_model

warnings.filterwarnings("ignore")

### Constantes e períodos
Datas do protocolo, custos, janelas do momentum (`6-1`, `9-1`, `12-1`), regras de seleção e `definir_periodos` (treino, validação e, se `FIM_DADOS` for estendido, pós-validação).

In [ ]:
# ----------------------------------------------------------------------------
# Constantes do notebook
# ----------------------------------------------------------------------------
INICIO_HISTORICO = "2015-01-01"
INICIO_BACKTEST = "2017-01-01"
FIM_DADOS = "2021-12-31"
CUSTOS = (0.0, 0.001, 0.003)
CUSTO_BASE = 0.001
JANELAS = {"6-1": 126, "9-1": 189, "12-1": 252}
SKIP = 21
SELECOES = ("Top 5", "Top 10", "Top 15", "Deadband 7", "Deadband 10")


def definir_periodos(fim_dados: str) -> dict:
    p = {"Treinamento (2017–2020)": ("2017-01-01", "2020-12-31"),
         "Validação (2021)": ("2021-01-01", "2021-12-31")}
    if pd.Timestamp(fim_dados) > pd.Timestamp("2021-12-31"):
        p["Pós-validação (2022+)"] = ("2022-01-01", fim_dados)
    return p

### Dados: planilha e CDI
`localizar_planilha` (adaptada), `load_data` (lê a planilha Economatica, trata duplicados) e `baixar_cdi` (série SGS 12 do Banco Central, com cache em disco).

In [ ]:
# ----------------------------------------------------------------------------
# 0) Dados: leitura da planilha, CDI
# ----------------------------------------------------------------------------
_TICKER_RE = re.compile(r"^[A-Z][A-Z0-9.\-]{1,9}$")


def localizar_planilha(nome="economatica-3.xlsx") -> Path:
    """Procura a planilha a partir da pasta do notebook, subindo pelas pastas-pai.
    Em cada pasta tenta: <pasta>/baseDeDados/<nome>, <pasta>/bases de dados/<nome> e <pasta>/<nome>.
    (O script original usava __file__, que não existe em notebooks.)"""
    pasta = Path.cwd().resolve()
    tentados = []
    for p in [pasta, *pasta.parents]:
        for sub in ("baseDeDados", "bases de dados", ""):
            c = (p / sub / nome) if sub else (p / nome)
            tentados.append(c)
            if c.is_file():
                return c
    raise FileNotFoundError(
        f"Planilha '{nome}' não encontrada a partir de {pasta}. "
        "Defina PLANILHA = r'caminho\\completo\\economatica-3.xlsx' na célula de parâmetros.\n"
        "Procurei, por exemplo, em: " + "; ".join(str(t) for t in tentados[:4]))


def load_data(path, sheet=0) -> pd.DataFrame:
    """Lê planilha Economatica (cabeçalho na linha 4); ticker = última linha do cabeçalho
    (ou nome da aba, se a coluna não trouxer ticker). Só colunas de fechamento; duplicados
    resolvidos mantendo a coluna de maior cobertura."""
    xl = pd.ExcelFile(path)
    sheet_name = xl.sheet_names[sheet] if isinstance(sheet, int) else sheet
    raw = xl.parse(sheet_name, header=None)
    header = raw.iloc[3].astype(str).tolist()[1:]

    tickers, is_price = [], []
    for h in header:
        lines = [x.strip() for x in h.strip().split("\n") if x.strip()]
        tickers.append(lines[-1] if lines and _TICKER_RE.match(lines[-1]) else str(sheet_name))
        is_price.append("fechamento" in h.lower())

    df = raw.iloc[4:, :].copy()
    df.columns = ["Data"] + tickers
    df["Data"] = pd.to_datetime(df["Data"], errors="coerce")
    df = df.dropna(subset=["Data"]).set_index("Data").sort_index()
    df = df.replace("-", np.nan).apply(pd.to_numeric, errors="coerce")
    df = df[~df.index.duplicated(keep="last")]

    if any(is_price) and not all(is_price):
        print(f"[dados] colunas que não são fechamento removidas: "
              f"{[t for t, ok in zip(tickers, is_price) if not ok]}")
        df = df.loc[:, is_price]
    cols = list(df.columns)
    if len(set(cols)) < len(cols):
        dups = sorted({c for c in cols if cols.count(c) > 1})
        best = {}
        for i, c in enumerate(cols):
            n = int(df.iloc[:, i].notna().sum())
            if c not in best or n > best[c][1]:
                best[c] = (i, n)
        print(f"[dados] tickers duplicados {dups}: mantida a coluna de maior cobertura de cada um")
        df = df.iloc[:, sorted(i for i, _ in best.values())]
    return df


def baixar_cdi(inicio: str, fim: str, cache: str = "cdi_sgs12.csv") -> pd.Series | None:
    """CDI diário (SGS 12 do BCB, em fração ao dia), como no notebook. Usa cache local;
    devolve None (com aviso) se não houver internet nem cache."""
    ini = pd.Timestamp(inicio) - pd.Timedelta(days=45)
    fim_ts = pd.Timestamp(fim)
    if os.path.exists(cache):
        s = pd.read_csv(cache, index_col=0, parse_dates=True).iloc[:, 0]
        if s.index.min() <= ini + pd.Timedelta(days=10) and s.index.max() >= fim_ts - pd.Timedelta(days=10):
            return s.loc[ini:fim_ts]
    try:
        import requests
        partes, a = [], ini
        while a <= fim_ts:  # janelas de 5 anos (a API limita a extensão das consultas)
            b = min(a + pd.DateOffset(years=5) - pd.Timedelta(days=1), fim_ts)
            r = requests.get("https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados",
                             params={"formato": "json", "dataInicial": a.strftime("%d/%m/%Y"),
                                     "dataFinal": b.strftime("%d/%m/%Y")}, timeout=30)
            r.raise_for_status()
            partes.append(pd.DataFrame(r.json()))
            a = b + pd.Timedelta(days=1)
        d = pd.concat(partes)
        d["data"] = pd.to_datetime(d["data"], format="%d/%m/%Y")
        s = (pd.to_numeric(d["valor"]) / 100).groupby(d["data"]).last().sort_index().rename("cdi")
        s.to_csv(cache)
        return s
    except Exception as e:
        print(f"[cdi] não foi possível obter o CDI ({type(e).__name__}); Sharpe será calculado "
              "sem taxa livre de risco. Passe `cdi=` com a série diária para corrigir.")
        return None

### Sinais de momentum
`calcular_momentum_mensal` (sinal no último pregão do mês), `criar_selecao` (Top N ou deadband) e `preparar_dados` (une tudo e devolve preços, retornos, sinais e datas).

In [ ]:
# ----------------------------------------------------------------------------
# 1) Sinais de momentum (idênticos ao notebook)
# ----------------------------------------------------------------------------
def calcular_momentum_mensal(precos, dias, skip=SKIP, inicio_bt=INICIO_BACKTEST, fim=FIM_DADOS):
    indicador = precos.shift(skip) / precos.shift(dias) - 1
    mensal = indicador.groupby(indicador.index.to_period("M")).tail(1)
    return mensal.loc[inicio_bt:fim]


def criar_selecao(momentum_mensal, nome: str):
    """'Top 5' -> criar_top_n(5); 'Deadband 7' -> criar_deadband(7, n=5)."""
    tipo, k = nome.split()
    return criar_top_n(momentum_mensal, int(k)) if tipo == "Top" else criar_deadband(momentum_mensal, int(k))


def preparar_dados(path=None, inicio_hist=INICIO_HISTORICO, inicio_bt=INICIO_BACKTEST,
                   fim=FIM_DADOS, exclude=()):
    path = path or localizar_planilha()
    df = load_data(path).drop(columns=list(exclude), errors="ignore")
    base = df.reset_index().rename(columns={df.index.name or "index": "Data"})
    precos = preparar_precos(base, inicio_hist, fim)           # função do notebook
    retornos = precos.pct_change(fill_method=None)
    momentos = {nome: calcular_momentum_mensal(precos, dias, SKIP, inicio_bt, fim)
                for nome, dias in JANELAS.items()}
    datas = momentos["12-1"].index
    print(f"[dados] {precos.shape[1]} ativos | {precos.index.min().date()} a {precos.index.max().date()}"
          f" | {len(datas)} datas de sinal")
    return precos, retornos, momentos, datas

### Previsão de volatilidade
Para cada ativo e data de sinal, ajusta o modelo só com retornos até o fechamento daquele dia e prevê a vol média dos próximos 21 pregões. `prever_vol` guarda o resultado em cache (`cache_vol_mensal.pkl`).

In [ ]:
# ----------------------------------------------------------------------------
# 2) Previsão de volatilidade nas datas de sinal (GARCH / GJR / EWMA)
# ----------------------------------------------------------------------------
def ewma_vol(r: pd.DataFrame, lam: float = 0.94) -> pd.DataFrame:
    out = {}
    for c in r.columns:
        x = r[c].dropna()
        out[c] = np.sqrt((x ** 2).ewm(alpha=1 - lam, adjust=False).mean())
    return pd.DataFrame(out).reindex(r.index).ffill(limit=5)


def garch_forecast(x: pd.Series, model: str, horizon: int, min_obs: int) -> float:
    x = x.dropna() * 100
    if len(x) < min_obs:
        return np.nan
    try:
        am = arch_model(x, mean="Zero", vol="GARCH", p=1, o=1 if model == "gjr" else 0, q=1,
                        dist="t", rescale=False)
        res = am.fit(disp="off", show_warning=False)
        if res.convergence_flag != 0:
            return np.nan
        var_path = res.forecast(horizon=horizon, reindex=False).variance.values[-1]
        return float(np.sqrt(var_path.mean()) / 100)
    except Exception:
        return np.nan


def _worker(args):
    a, r, ew_fb, ew_spec, pos, model, param, horizon, min_asset_obs = args
    out = np.full(len(pos), np.nan)
    n_fb = 0
    w = int(param) if model != "ewma" else 756
    for k, p in enumerate(pos):
        x = r[max(0, p + 1 - w): p + 1]                 # só dados até a data do sinal
        if np.isfinite(x).sum() < min_asset_obs:
            continue
        if model == "ewma":
            out[k] = ew_spec[p]
            continue
        s = garch_forecast(pd.Series(x), model, horizon, min_asset_obs)
        if np.isfinite(s):
            out[k] = s
        else:
            out[k] = ew_fb[p]; n_fb += 1                 # fallback EWMA(0.94)
    return a, out, n_fb


def prever_vol(retornos, datas, specs, horizonte=21, min_obs_ativo=252, workers=None,
               cache_path="cache_vol_mensal.pkl"):
    """Vol diária prevista (média do horizonte) nas `datas`, para cada especificação em `specs`
    (lista de (modelo, parâmetro): ('gjr', 756), ('garch', 504), ('ewma', 0.94)).
    Retorna ({spec: DataFrame[datas x ativos]}, {spec: nº de fallbacks p/ EWMA}).
    Cache incremental em disco (retomável)."""
    logr = np.log1p(retornos)
    pos = [logr.index.get_loc(t) for t in datas]
    fp = (tuple(logr.columns), str(logr.index[0]), str(logr.index[-1]),
          len(datas), str(datas[0]), str(datas[-1]), horizonte, min_obs_ativo)

    cache = {}
    if cache_path and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            c = pickle.load(f)
        if c.get("fingerprint") == fp:
            cache = c["specs"]
            print(f"[vol] cache: {sum(s in cache for s in specs)}/{len(specs)} especificações prontas")

    ew_fb = ewma_vol(logr, 0.94)
    n = workers or os.cpu_count() or 1
    for spec in specs:
        if spec in cache:
            continue
        model, param = spec
        t0 = time.time()
        ew_spec = ewma_vol(logr, param) if model == "ewma" else ew_fb
        tasks = [(a, logr[a].values, ew_fb[a].values, ew_spec[a].values, pos, model, param,
                  horizonte, min_obs_ativo) for a in logr.columns]
        if n > 1:
            with ProcessPoolExecutor(min(n, len(tasks))) as ex:
                res = list(ex.map(_worker, tasks))
        else:
            res = [_worker(t) for t in tasks]
        df = pd.DataFrame(index=datas, columns=logr.columns, dtype=float)
        for a, arr, _ in res:
            df[a] = arr
        cache[spec] = (df, sum(r[2] for r in res))
        if cache_path:
            with open(cache_path, "wb") as f:
                pickle.dump({"fingerprint": fp, "specs": cache}, f)
        print(f"[vol] {model}-{param} pronto em {time.time() - t0:.0f}s", flush=True)
    return {s: cache[s][0] for s in specs}, {s: cache[s][1] for s in specs}

### Pesos e backtests
`criar_pesos_vol` (peso ∝ 1/σ̂, soma 1) e `executar_backtests`, que roda cada combinação de janela, seleção e esquema de pesos com a mesma `backtest` do notebook.

In [ ]:
# ----------------------------------------------------------------------------
# 3) Pesos: iguais (notebook) vs inverso da vol prevista (GARCH)
# ----------------------------------------------------------------------------
def criar_pesos_vol(carteira, sigma, universo):
    """Mesmos ativos de `carteira`, com peso ∝ 1/σ̂ (som a 1). Se algum σ̂ faltar, usa a mediana
    dos σ̂ disponíveis da carteira (ou pesos iguais se nenhum existir)."""
    pesos = pd.DataFrame(0.0, index=carteira.index, columns=universo)
    n_faltas = 0
    for data, linha in carteira.iterrows():
        ativos = linha.dropna().tolist()
        if not ativos or len(set(ativos)) != len(ativos):
            raise ValueError(f"Carteira vazia ou com duplicatas em {data}.")
        s = sigma.loc[data, ativos].astype(float).where(lambda x: x > 0)
        if s.isna().any():
            n_faltas += int(s.isna().sum())
            s = s.fillna(s.median() if s.notna().any() else 1.0)
        w = 1.0 / s
        pesos.loc[data, ativos] = (w / w.sum()).values
    if n_faltas:
        print(f"[pesos] {n_faltas} σ̂ ausentes substituídos pela mediana da carteira")
    return pesos


def nome_esquema(spec) -> str:
    return "igual" if spec is None else f"{spec[0]}-{spec[1]}"


def executar_backtests(retornos, momentos, sigmas, janelas, selecoes, esquemas,
                       inicio_bt=INICIO_BACKTEST, fim=FIM_DADOS):
    """esquemas: lista com None (pesos iguais, notebook) e/ou specs de vol.
    Retorna {(janela, seleção, esquema): DataFrame[Retorno, Turnover]} e as carteiras."""
    resultados, carteiras = {}, {}
    for jn in janelas:
        for sel in selecoes:
            carteira = criar_selecao(momentos[jn], sel)
            carteiras[(jn, sel)] = carteira
            for esq in esquemas:
                pesos = (criar_pesos(carteira, retornos.columns) if esq is None
                         else criar_pesos_vol(carteira, sigmas[esq], retornos.columns))
                resultados[(jn, sel, nome_esquema(esq))] = backtest(pesos, retornos, inicio_bt, fim)
    return resultados, carteiras

### Relatório e função `run`
`tabela_metricas`, `plot_patrimonio` e `run`, que executa uma configuração completa e compara pesos iguais com pesos por volatilidade.

In [ ]:
# ----------------------------------------------------------------------------
# 4) Relatório (mesmas métricas do notebook)
# ----------------------------------------------------------------------------
def tabela_metricas(resultados, periodos, cdi=None, custo=CUSTO_BASE):
    tabs = {}
    for per, (ini, fim) in periodos.items():
        tabs[per] = pd.DataFrame({k: calcular_metricas(r.loc[ini:fim], custo, cdi=cdi)
                                  for k, r in resultados.items()}).T
    return tabs


def plot_patrimonio(resultados, periodos, caminho, janela):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cores = {s: f"C{i}" for i, s in enumerate(SELECOES)}
    fig, ax = plt.subplots(1, len(periodos), figsize=(7 * len(periodos), 5))
    ax = np.atleast_1d(ax)
    for a, (per, (ini, fim)) in zip(ax, periodos.items()):
        for (jn, sel, esq), r in resultados.items():
            pat = (1 + aplicar_custos(r.loc[ini:fim], CUSTO_BASE)).cumprod()
            a.plot(pat.index, pat.values, color=cores[sel], ls="--" if esq == "igual" else "-",
                   lw=1.2, label=f"{sel} | {esq}")
        a.set_title(f"{per} — janela {janela}, custo {CUSTO_BASE:.2%}")
        a.set_ylabel("Patrimônio (base 1)"); a.grid(alpha=.3)
    ax[0].legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.savefig(caminho, dpi=130); plt.close()


def run(path=None, janela="12-1", vol=("gjr", 756), cdi="auto", exclude=(),
        fim=FIM_DADOS, plot_path=None, workers=None, cache_path="cache_vol_mensal.pkl"):
    precos, retornos, momentos, datas = preparar_dados(path, INICIO_HISTORICO, INICIO_BACKTEST, fim, exclude)
    periodos = definir_periodos(fim)
    if isinstance(cdi, str) and cdi == "auto":
        cdi = baixar_cdi(INICIO_BACKTEST, fim)

    sigmas, fb = prever_vol(retornos, datas, [vol], workers=workers, cache_path=cache_path)
    print(f"[vol] {nome_esquema(vol)}: {fb[vol]} previsões caíram para EWMA")

    resultados, carteiras = executar_backtests(retornos, momentos, sigmas, [janela], SELECOES,
                                               [None, vol], INICIO_BACKTEST, fim)
    tabs = tabela_metricas(resultados, periodos, cdi)
    for per, t in tabs.items():
        print(f"\n{per} — custo {CUSTO_BASE:.2%}" + ("" if cdi is not None else "  [Sharpe sem CDI]"))
        print(t.round(3).to_string())

    print("\nΔ Sharpe (pesos por vol − pesos iguais), mesma seleção:")
    for per, t in tabs.items():
        d = {sel: t.loc[(janela, sel, nome_esquema(vol)), "Sharpe"] - t.loc[(janela, sel, "igual"), "Sharpe"]
             for sel in SELECOES}
        print(f"  {per}: " + " | ".join(f"{k}: {v:+.2f}" for k, v in d.items()))
    if plot_path:
        plot_patrimonio(resultados, periodos, plot_path, janela)
    return {"resultados": resultados, "carteiras": carteiras, "metricas": tabs,
            "sigmas": sigmas, "precos": precos, "retornos": retornos}

---
## Parte 3 — Otimização (`momentum-garch-otimizacao.py`)

Descrição original do arquivo:

```text
Busca de parâmetros do Momentum 12-1 + GARCH, com o PROTOCOLO DO NOTEBOOK momentums.ipynb:
  - parâmetros escolhidos SÓ no treino (2017–2020); a validação (2021) apenas reporta;
  - sinal, seleção, execução, custos e métricas = funções do notebook (funcoes_momentum.py).

Grade (cada configuração é um backtest completo):
  janela do momentum   : 6-1, 9-1, 12-1                       (3)
  seleção              : Top 5/10/15, Deadband 7/10           (5)
  esquema de pesos     : iguais (notebook)  +  1/σ̂ com σ̂ de   (1 + 9)
                         GARCH(504/756/1008), GJR(504/756/1008), EWMA(λ 0.90/0.94/0.97)
  => 150 backtests (15 com pesos iguais e 45 por método de vol).

Comparação coerente entre métodos de vol: mesma seleção de ativos, mesmas datas, mesmo custo,
grades do mesmo tamanho (3 especificações por método). Só muda a previsão de σ̂ que
dimensiona os pesos. A seleção por ranking não usa σ̂, então a vol só afeta o tamanho.

Relatórios:
  1. melhor configuração de cada família escolhida no TREINO e seu resultado na VALIDAÇÃO;
  2. efeito do dimensionamento por vol: Δ Sharpe (pesos por vol − pesos iguais) com a MESMA
     seleção, por método e período;
  3. qualidade da previsão de vol (QLIKE) nas datas de sinal.

Requer funcoes_momentum.py e momentum_garch_dados_teste.py na mesma pasta.
```

### Famílias de volatilidade
GARCH, GJR e EWMA, com 3 especificações cada (grades do mesmo tamanho) e o mapa de cada esquema para sua família.

In [ ]:
import numpy as np
import pandas as pd

METHODS = {
    "GARCH": [("garch", 504), ("garch", 756), ("garch", 1008)],
    "GJR":   [("gjr", 504), ("gjr", 756), ("gjr", 1008)],
    "EWMA":  [("ewma", 0.90), ("ewma", 0.94), ("ewma", 0.97)],
}
VOL_SPECS = [s for specs in METHODS.values() for s in specs]
FAMILIA = {nome_esquema(s): m for m, specs in METHODS.items() for s in specs}
FAMILIA["igual"] = "Pesos iguais"
ORDEM_FAMILIAS = ["Pesos iguais", "GARCH", "GJR", "EWMA"]

### Grade e relatórios
`montar_grade` (métricas por configuração e período), `melhores_por_familia` (escolhe no treino, reporta na validação), `efeito_do_dimensionamento` (Δ Sharpe contra pesos iguais), `qlike_por_spec` (qualidade da previsão de vol) e `plot_melhores`.

In [ ]:
# ----------------------------------------------------------------------------
# Grade de resultados
# ----------------------------------------------------------------------------
def montar_grade(resultados, periodos, cdi) -> pd.DataFrame:
    """Linhas = (janela, seleção, esquema); colunas = (período, métrica)."""
    tabs = tabela_metricas(resultados, periodos, cdi, CUSTO_BASE)
    grade = pd.concat(tabs, axis=1)            # colunas MultiIndex (período, métrica)
    grade.index = pd.MultiIndex.from_tuples(grade.index, names=["janela", "seleção", "esquema"])
    grade["família"] = [FAMILIA[e] for e in grade.index.get_level_values("esquema")]
    return grade


def melhores_por_familia(grade, p_treino, p_valid):
    """Escolhe, em cada família, a configuração de maior Sharpe no TREINO e mostra a validação."""
    linhas = {}
    for fam in ORDEM_FAMILIAS:
        g = grade[grade["família"] == fam]
        melhor = g[(p_treino, "Sharpe")].idxmax()
        linhas[fam] = {
            "configuração": " | ".join(map(str, melhor)),
            "Sharpe treino": g.loc[melhor, (p_treino, "Sharpe")],
            "Sharpe validação": g.loc[melhor, (p_valid, "Sharpe")],
            "Retorno val. (%)": g.loc[melhor, (p_valid, "Retorno anual (%)")],
            "Vol val. (%)": g.loc[melhor, (p_valid, "Volatilidade (%)")],
            "MaxDD val. (%)": g.loc[melhor, (p_valid, "Max Drawdown (%)")],
            "Sharpe grade (mediana) treino": g[(p_treino, "Sharpe")].median(),
            "Sharpe grade (mediana) val.": g[(p_valid, "Sharpe")].median(),
            "nº configs": len(g),
        }
    return pd.DataFrame(linhas).T, {f: grade[grade["família"] == f][(p_treino, "Sharpe")].idxmax()
                                    for f in ORDEM_FAMILIAS}


def efeito_do_dimensionamento(grade, periodos):
    """Δ Sharpe = Sharpe(pesos por vol) − Sharpe(pesos iguais), mesma (janela, seleção)."""
    sh = {per: grade[(per, "Sharpe")].unstack("esquema") for per in periodos}
    linhas = {}
    for fam, specs in METHODS.items():
        cols = [nome_esquema(s) for s in specs]
        linha = {}
        for per in periodos:
            d = sh[per][cols].sub(sh[per]["igual"], axis=0).stack()
            linha[f"mediana Δ {per.split()[0]}"] = d.median()
            linha[f"% Δ>0 {per.split()[0]}"] = (d > 0).mean() * 100
        linhas[fam] = linha
    return pd.DataFrame(linhas).T


def qlike_por_spec(retornos, sigmas, datas, horizonte=21):
    logr = np.log1p(retornos)
    rv = (logr ** 2).rolling(horizonte, min_periods=horizonte // 2).mean().shift(-horizonte).reindex(datas)
    out = {}
    for spec, sg in sigmas.items():
        fc = sg ** 2
        m = (rv > 0) & (fc > 0)
        ratio = (rv / fc).where(m)
        out[nome_esquema(spec)] = float((ratio - np.log(ratio) - 1).stack().mean())
    return pd.Series(out, name="QLIKE (menor = melhor)")


def plot_melhores(resultados, escolhidos, periodos, caminho):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, len(periodos), figsize=(7 * len(periodos), 5))
    ax = np.atleast_1d(ax)
    for a, (per, (ini, fim)) in zip(ax, periodos.items()):
        for fam, chave in escolhidos.items():
            pat = (1 + aplicar_custos(resultados[chave].loc[ini:fim], CUSTO_BASE)).cumprod()
            a.plot(pat.index, pat.values, label=f"{fam}: {' | '.join(map(str, chave))}",
                   ls="--" if fam == "Pesos iguais" else "-")
        a.set_title(f"{per}\n(configurações escolhidas no treino)"); a.grid(alpha=.3)
        a.set_ylabel("Patrimônio (base 1)")
    ax[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(caminho, dpi=130); plt.close()

### `run_optimization`
Executa a grade completa (150 backtests), imprime os três relatórios e salva figura e CSVs.

In [ ]:
# ----------------------------------------------------------------------------
# Execução
# ----------------------------------------------------------------------------
def run_optimization(path=None, cdi="auto", exclude=(), fim=FIM_DADOS, workers=None,
                     cache_path="cache_vol_mensal.pkl", saida_prefixo="otimizacao"):
    precos, retornos, momentos, datas = preparar_dados(path, exclude=exclude, fim=fim)
    periodos = definir_periodos(fim)
    p_treino, p_valid = list(periodos)[:2]
    if isinstance(cdi, str) and cdi == "auto":
        cdi = baixar_cdi(INICIO_BACKTEST, fim)
    sem_cdi = "" if cdi is not None else "  [Sharpe sem CDI]"

    sigmas, fallbacks = prever_vol(retornos, datas, VOL_SPECS, workers=workers, cache_path=cache_path)
    print("[vol] previsões GARCH/GJR que caíram para EWMA:",
          {nome_esquema(s): fallbacks[s] for s in VOL_SPECS if s[0] != "ewma"})

    resultados, _ = executar_backtests(retornos, momentos, sigmas, JANELAS, SELECOES,
                                       [None] + VOL_SPECS, INICIO_BACKTEST, fim)
    print(f"[grade] {len(resultados)} backtests executados")
    grade = montar_grade(resultados, periodos, cdi)

    tab, escolhidos_idx = melhores_por_familia(grade, p_treino, p_valid)
    print(f"\n1) Melhor configuração de cada família (escolhida no treino){sem_cdi}")
    print(tab.drop(columns=["configuração"]).round(3).to_string())
    print("   configurações:", {f: tab.loc[f, "configuração"] for f in ORDEM_FAMILIAS})

    efeito = efeito_do_dimensionamento(grade, periodos)
    print(f"\n2) Efeito de dimensionar por vol (Δ Sharpe vs pesos iguais, mesma seleção; "
          f"{len(JANELAS) * len(SELECOES) * 3} pares por método){sem_cdi}")
    print(efeito.round(3).to_string())

    ql = qlike_por_spec(retornos, sigmas, datas)
    print("\n3) Qualidade da previsão de vol nas datas de sinal (QLIKE)")
    print(ql.round(4).to_string())

    plot_melhores(resultados, escolhidos_idx, periodos, f"{saida_prefixo}_melhores.png")
    g = grade.copy(); g.columns = [" | ".join(c) if isinstance(c, tuple) and c[1] else c[0] for c in g.columns]
    g.round(4).to_csv(f"{saida_prefixo}_grade.csv")
    tab.round(4).to_csv(f"{saida_prefixo}_familias.csv")
    return {"grade": grade, "familias": tab, "efeito": efeito, "qlike": ql, "resultados": resultados}

---
## Parte 4 — Executar

### Parâmetros de execução

In [ ]:
# Caminho da planilha. None = procura automaticamente (baseDeDados/ ou bases de dados/, na pasta atual ou acima).
# Se a busca falhar, informe o caminho completo, por exemplo:
#   PLANILHA = r"C:\projeto\baseDeDados\economatica-3.xlsx"
PLANILHA = None

# Processos para o GARCH. Em notebooks no Windows use 1 (a versão paralela precisa de um arquivo .py importável).
WORKERS = 1

### (Opcional) Baixar e guardar o CDI

Faz a consulta ao Banco Central (SGS 12) uma vez e grava `cdi_sgs12.csv`. As execuções seguintes usam o arquivo, sem internet.
Sem o CDI, o Sharpe é calculado **sem taxa livre de risco** e as tabelas avisam com `[Sharpe sem CDI]`.

In [ ]:
cdi = baixar_cdi("2017-01-01", "2026-10-31")
if cdi is not None:
    print(f"CDI: {len(cdi)} datas, de {cdi.index.min().date()} a {cdi.index.max().date()}")

### 4.1 Uma configuração: Top N / Deadband × pesos iguais vs GJR-756

Para cada regra de seleção (`Top 5`, `Top 10`, `Top 15`, `Deadband 7`, `Deadband 10`), compara pesos iguais com pesos por 1/σ̂ do GJR (janela de 756 pregões), usando a janela de momentum 12-1.

In [ ]:
out = run(path=PLANILHA, janela="12-1", vol=("gjr", 756), workers=WORKERS,
          plot_path="momentum_garch_resultado.png")

In [ ]:
%matplotlib inline
for periodo, tabela in out["metricas"].items():
    print(periodo)
    display(tabela.round(3))
display(Image("momentum_garch_resultado.png"))

## Parte 5 — Grade completa (150 configurações)

3 janelas × 5 seleções × 10 esquemas de pesos (iguais + 3 de GARCH + 3 de GJR + 3 de EWMA). A melhor configuração de cada família é escolhida **só no treino**, e a validação apenas reporta. Na primeira execução, ajusta as especificações de volatilidade que ainda não estão em cache.

In [ ]:
res = run_optimization(path=PLANILHA, workers=WORKERS, saida_prefixo="otimizacao")

In [ ]:
%matplotlib inline
print("1) Melhor configuração de cada família (escolhida no treino)")
display(res["familias"].drop(columns=["configuração"]).round(3))
print("   Configurações:", dict(res["familias"]["configuração"]))

print("\n2) Efeito de dimensionar por vol (Δ Sharpe vs pesos iguais, mesma seleção)")
display(res["efeito"].round(3))

print("\n3) Qualidade da previsão de volatilidade (QLIKE; menor é melhor)")
display(res["qlike"].round(4).to_frame())

display(Image("otimizacao_melhores.png"))

A grade completa (todas as métricas, por configuração e período) fica em `otimizacao_grade.csv`, e o resumo por família em `otimizacao_familias.csv`.
Também está em `res["grade"]`:

In [ ]:
res["grade"].round(3).head(15)